## Slide 2 — Load and inspect the dataset
If running in **Google Colab**, upload the Excel file when prompted. In local Jupyter, place the workbook in the same folder as this notebook or edit `FILE_PATH`.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from IPython.display import display

FILE_PATH = Path("DAV_Dataset_New.xlsx")

xls = pd.ExcelFile(FILE_PATH)
print("Workbook sheets:", xls.sheet_names)

df_raw = pd.read_excel(FILE_PATH, sheet_name="Sales_Data")
print("Shape:", df_raw.shape)
display(df_raw.head(10))

Workbook sheets: ['Sales_Data', 'Instructor_Error_Summary']
Shape: (500, 10)


,Order_ID,Month,Month_Number,Category,Region,Channel,Units_Sold,Sales_INR,Discount_Percent,Customer_Rating
0,1.0,Nov,11.0,Books,West,Online,30.0,17072.0,10.0,3.9
1,2.0,Aug,8.0,Furniture,North,Store,33.0,31283.0,15.0,4.1
2,3.0,Dec,12.0,Clothing,North,Online,85.0,30146.0,5.0,3.8
3,4.0,Dec,12.0,Books,West,Online,71.0,18879.0,15.0,3.6
4,5.0,Sep,9.0,Books,East,Online,60.0,15920.0,5.0,5.0
5,6.0,Sep,9.0,Books,West,Store,18.0,21085.0,15.0,3.4
6,7.0,Aug,8.0,Stationery,East,Online,87.0,8850.0,15.0,NaN
7,8.0,Jun,6.0,Clothing,South,Store,13.0,31443.0,15.0,4.5
8,463.5,Aug,8.0,Electronics,North,Store,91.0,63628.0,0.0,4.6
9,10.0,Dec,12.0,Books,West,Online,50.0,14660.0,0.0,3.7


In [ ]:
print("Column names:", df_raw.columns.tolist())
display(df_raw.info())
display(df_raw.describe(include="all").T)
print("\\nMissing values per column:")
display(df_raw.isna().sum().to_frame("Missing count"))

if "Instructor_Error_Summary" in xls.sheet_names:
    error_summary = pd.read_excel(FILE_PATH, sheet_name="Instructor_Error_Summary")
    print("Instructor's data-quality checklist:")
    display(error_summary)

## Slide 3 — Dashboard concepts and KPI design
An **interactive dashboard** combines key performance indicators (KPIs), visualizations, and controls in one interface. Users can change selections and explore different views without manually creating a new report.

For this dataset, useful metrics include:
- **Total Sales (INR):** sum of `Sales_INR`
- **Units Sold:** sum of `Units_Sold`
- **Average Customer Rating:** mean of `Customer_Rating`
- **Average Discount (%):** mean of `Discount_Percent`

The file does not contain a verified order-count or profit column, so this notebook does not invent those metrics.

## Slide 4 — Static chart versus interactive chart
A static chart is a fixed view. An interactive Plotly chart supports hover details, zooming, and legend selection. Dashboard frameworks add controls that filter the data and update outputs.

In [ ]:
# A simple static Matplotlib chart (will use cleaned data after Slide 6).
# This example uses the raw data only to illustrate a fixed chart.
static_demo = df_raw.dropna(subset=["Region", "Sales_INR"]).copy()
static_region_sales = static_demo.groupby("Region", dropna=False)["Sales_INR"].sum().sort_values()

plt.figure(figsize=(8, 4))
static_region_sales.plot(kind="barh")
plt.title("Static view: Sales by region")
plt.xlabel("Sales (INR)")
plt.ylabel("Region")
plt.tight_layout()
plt.show()

## Slide 5 — Dash and Shiny for Python
**Dash** (Plotly) uses a component-based layout and callbacks to connect inputs to outputs. It is a natural fit for dashboards built around Plotly charts.

**Shiny for Python** uses reactive programming: outputs automatically respond to changes in input values. It is useful for interactive data apps and rapid exploration.

In this session, Dash receives the main implementation. Shiny is shown as a compact equivalent to ensure both syllabus frameworks are covered.

## Slide 6 — Clean the data and design KPIs
Cleaning decisions should be transparent. Here we:
1. Standardize text labels (for example, `NORTH` → `North`).
2. Convert numeric fields to numeric types.
3. Remove exact duplicate rows, while reporting the impact.
4. Fill missing categorical values with the mode and numeric values with the median for this teaching example.
5. Standardize month names and derive a month number.

**Important:** In a real business workflow, verify whether repeated rows are true duplicates before removing them. The dataset includes repeated records by design, so compare the row counts before and after cleaning.

In [ ]:
df = df_raw.copy()
rows_before = len(df)
missing_before = int(df.isna().sum().sum())
exact_duplicates_before = int(df.duplicated().sum())

# Normalize whitespace and case in text columns
text_cols = df.select_dtypes(include=["object", "string"]).columns
for col in text_cols:
    df[col] = df[col].astype("string").str.strip()

# Standardize common categorical columns with title case
for col in ["Category", "Region", "Channel"]:
    if col in df.columns:
        df[col] = df[col].str.title()

# Standardize month names to three-letter abbreviations
if "Month" in df.columns:
    df["Month"] = df["Month"].str.title().str.slice(0, 3)

numeric_cols = [
    c for c in ["Order_ID", "Month_Number", "Units_Sold", "Sales_INR",
                "Discount_Percent", "Customer_Rating"]
    if c in df.columns
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove exact duplicate rows only
df = df.drop_duplicates().copy()
rows_after_dedup = len(df)

# Fill missing values for a classroom-ready demonstration
for col in df.select_dtypes(include=["object", "string"]).columns:
    if df[col].isna().any():
        mode_values = df[col].mode(dropna=True)
        if not mode_values.empty:
            df[col] = df[col].fillna(mode_values.iloc[0])

for col in df.select_dtypes(include=["number"]).columns:
    if df[col].isna().any():
        median_value = df[col].median()
        if pd.notna(median_value):
            df[col] = df[col].fillna(median_value)

# Ensure month number is consistent with month text
month_map = {
    "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "Jun": 6,
    "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12
}
if "Month" in df.columns:
    df["Month"] = df["Month"].map(lambda x: month_map.get(str(x)[:3].title(), x))
    if "Month_Number" in df.columns:
        df["Month_Number"] = df["Month"].map(month_map).fillna(df["Month_Number"])

df = df.dropna(subset=["Sales_INR", "Region", "Category"]).copy()

print(f"Rows at start: {rows_before}")
print(f"Exact duplicate rows detected: {exact_duplicates_before}")
print(f"Rows after exact-duplicate removal: {rows_after_dedup}")
print(f"Rows after classroom cleaning: {len(df)}")
print(f"Missing cells before cleaning: {missing_before}")
print(f"Missing cells after cleaning: {int(df.isna().sum().sum())}")
display(df.head())

In [ ]:
# KPI calculations
total_sales = df["Sales_INR"].sum()
total_units = df["Units_Sold"].sum() if "Units_Sold" in df else np.nan
avg_rating = df["Customer_Rating"].mean() if "Customer_Rating" in df else np.nan
avg_discount = df["Discount_Percent"].mean() if "Discount_Percent" in df else np.nan

print(f"Total Sales: ₹{total_sales:,.0f}")
print(f"Units Sold: {total_units:,.0f}" if pd.notna(total_units) else "Units Sold: N/A")
print(f"Average Customer Rating: {avg_rating:.2f} / 5" if pd.notna(avg_rating) else "Average Customer Rating: N/A")
print(f"Average Discount: {avg_discount:.1f}%" if pd.notna(avg_discount) else "Average Discount: N/A")

region_sales = df.groupby("Region", as_index=False)["Sales_INR"].sum().sort_values("Sales_INR", ascending=False)
category_sales = df.groupby("Category", as_index=False)["Sales_INR"].sum().sort_values("Sales_INR", ascending=False)
channel_sales = df.groupby("Channel", as_index=False)["Sales_INR"].sum().sort_values("Sales_INR", ascending=False)

display(region_sales)
display(category_sales)
display(channel_sales)

In [ ]:
# Choose charts based on the question
fig = px.bar(region_sales, x="Region", y="Sales_INR",
             title="Total Sales by Region",
             labels={"Sales_INR": "Sales (INR)", "Region": "Region"},
             color="Region")
fig.show()

monthly_sales = df.groupby("Month_Number", as_index=False)["Sales_INR"].sum().sort_values("Month_Number")
month_labels = {v: k for k, v in month_map.items()}
monthly_sales["Month"] = monthly_sales["Month_Number"].map(month_labels)
fig = px.line(monthly_sales, x="Month", y="Sales_INR", markers=True,
              title="Sales by Month",
              labels={"Sales_INR": "Sales (INR)", "Month": "Month"})
fig.show()

## Slide 7 — Build an interactive dashboard using Dash
The following cell creates a dashboard with a region dropdown, a category dropdown, KPI cards, and linked charts. It is designed for **local Jupyter Notebook** with a recent Dash installation. If the app does not render inline, follow the printed URL or run the same code as a Python script.

In [ ]:
%pip -q install dash plotly pandas openpyxl

In [ ]:
from dash import Dash, dcc, html, Input, Output
import plotly.express as px

# Use a fresh copy so dashboard callbacks never modify the cleaned source.
dash_df = df.copy()
regions = ["All Regions"] + sorted(dash_df["Region"].dropna().unique().tolist())
categories = ["All Categories"] + sorted(dash_df["Category"].dropna().unique().tolist())

app = Dash(__name__)
app.title = "SRU Sales Dashboard"

BLUE = "#0756A6"
ORANGE = "#F47721"

def kpi_card(label, value, color):
    return html.Div([
        html.Div(label, style={"fontSize": "14px", "color": "#334155"}),
        html.Div(value, style={"fontSize": "24px", "fontWeight": "bold", "color": color})
    ], style={"background": "white", "borderLeft": f"5px solid {color}",
              "padding": "14px", "borderRadius": "8px", "boxShadow": "0 1px 5px #d8e2ef",
              "flex": "1", "minWidth": "150px"})

app.layout = html.Div([
    html.H1("Interactive Sales Dashboard", style={"color": BLUE}),
    html.P("Explore sales by region and product category."),
    html.Div([
        html.Div([
            html.Label("Select Region"),
            dcc.Dropdown(id="region-filter", options=[{"label": x, "value": x} for x in regions],
                         value="All Regions", clearable=False)
        ], style={"flex": "1", "minWidth": "220px"}),
        html.Div([
            html.Label("Select Category"),
            dcc.Dropdown(id="category-filter", options=[{"label": x, "value": x} for x in categories],
                         value="All Categories", clearable=False)
        ], style={"flex": "1", "minWidth": "220px"})
    ], style={"display": "flex", "gap": "18px", "flexWrap": "wrap", "marginBottom": "20px"}),
    html.Div(id="kpi-row", style={"display": "flex", "gap": "14px", "flexWrap": "wrap", "marginBottom": "20px"}),
    html.Div([
        dcc.Graph(id="region-chart"),
        dcc.Graph(id="month-chart"),
        dcc.Graph(id="channel-chart"),
        dcc.Graph(id="category-chart")
    ], style={"display": "grid", "gridTemplateColumns": "repeat(auto-fit, minmax(320px, 1fr))", "gap": "12px"})
], style={"fontFamily": "Arial, sans-serif", "background": "#f4f8fc", "padding": "24px"})

@app.callback(
    Output("kpi-row", "children"),
    Output("region-chart", "figure"),
    Output("month-chart", "figure"),
    Output("channel-chart", "figure"),
    Output("category-chart", "figure"),
    Input("region-filter", "value"),
    Input("category-filter", "value")
)
def update_dashboard(selected_region, selected_category):
    filtered = dash_df.copy()
    if selected_region != "All Regions":
        filtered = filtered[filtered["Region"] == selected_region]
    if selected_category != "All Categories":
        filtered = filtered[filtered["Category"] == selected_category]

    sales = filtered["Sales_INR"].sum()
    units = filtered["Units_Sold"].sum() if "Units_Sold" in filtered else 0
    rating = filtered["Customer_Rating"].mean() if "Customer_Rating" in filtered else np.nan
    discount = filtered["Discount_Percent"].mean() if "Discount_Percent" in filtered else np.nan

    cards = [
        kpi_card("Total Sales", f"₹{sales:,.0f}", BLUE),
        kpi_card("Units Sold", f"{units:,.0f}", ORANGE),
        kpi_card("Avg. Rating", f"{rating:.2f}/5" if pd.notna(rating) else "N/A", BLUE),
        kpi_card("Avg. Discount", f"{discount:.1f}%" if pd.notna(discount) else "N/A", ORANGE)
    ]

    by_region = filtered.groupby("Region", as_index=False)["Sales_INR"].sum().sort_values("Sales_INR", ascending=False)
    by_month = filtered.groupby("Month_Number", as_index=False)["Sales_INR"].sum().sort_values("Month_Number")
    by_month["Month"] = by_month["Month_Number"].map(month_labels)
    by_channel = filtered.groupby("Channel", as_index=False)["Sales_INR"].sum()
    by_category = filtered.groupby("Category", as_index=False)["Sales_INR"].sum().sort_values("Sales_INR", ascending=False)

    fig_region = px.bar(by_region, x="Region", y="Sales_INR", title="Sales by Region", color="Region")
    fig_month = px.line(by_month, x="Month", y="Sales_INR", title="Sales by Month", markers=True)
    fig_channel = px.pie(by_channel, names="Channel", values="Sales_INR", title="Sales by Channel", hole=0.35)
    fig_category = px.bar(by_category, x="Category", y="Sales_INR", title="Sales by Category", color="Category")
    for fig in [fig_region, fig_month, fig_channel, fig_category]:
        fig.update_layout(margin=dict(l=20, r=20, t=50, b=20), template="plotly_white")
    return cards, fig_region, fig_month, fig_channel, fig_category

print("Dash app is defined. Run the next cell to launch it.")

In [ ]:
# Launch in Jupyter Notebook (Dash 2.11+ supports notebook modes).
# If inline mode is unsupported in your environment, use jupyter_mode="external"
# or save this app code as a .py file and run: python app.py
app.run(debug=False, jupyter_mode="inline")

## Slide 8 — Build a minimal Shiny for Python app
Shiny for Python uses a UI definition and reactive server functions. The example below writes a small app to `app_shiny.py`. Run it in a terminal or notebook shell after installing Shiny. This is a separate web app, not a cell output like a Plotly chart.

In [ ]:
%pip -q install shiny

In [ ]:
shiny_app_code = r'''
from shiny import App, ui, render, reactive
import pandas as pd
import plotly.express as px

DATA_FILE = "DAV_Dataset_New(7).xlsx"
data = pd.read_excel(DATA_FILE, sheet_name="Sales_Data")
for col in ["Region", "Category", "Channel"]:
    data[col] = data[col].astype("string").str.strip().str.title()
data["Sales_INR"] = pd.to_numeric(data["Sales_INR"], errors="coerce")
data = data.dropna(subset=["Region", "Category", "Sales_INR"]).drop_duplicates()

regions = ["All Regions"] + sorted(data["Region"].dropna().unique().tolist())
categories = ["All Categories"] + sorted(data["Category"].dropna().unique().tolist())

app_ui = ui.page_fluid(
    ui.h2("Interactive Sales Dashboard — Shiny for Python"),
    ui.layout_sidebar(
        ui.sidebar(
            ui.input_select("region", "Select region", choices=regions, selected="All Regions"),
            ui.input_select("category", "Select category", choices=categories, selected="All Categories")
        ),
        ui.output_text("sales_summary"),
        ui.output_plot("sales_chart")
    )
)

def server(input, output, session):
    @reactive.calc
    def filtered_data():
        d = data.copy()
        if input.region() != "All Regions":
            d = d[d["Region"] == input.region()]
        if input.category() != "All Categories":
            d = d[d["Category"] == input.category()]
        return d

    @output
    @render.text
    def sales_summary():
        d = filtered_data()
        return f"Total sales: ₹{d['Sales_INR'].sum():,.0f} | Rows: {len(d):,}"

    @output
    @render.plot
    def sales_chart():
        d = filtered_data()
        chart_data = d.groupby("Category", as_index=False)["Sales_INR"].sum()
        fig = px.bar(chart_data, x="Category", y="Sales_INR",
                     title="Sales by Category", color="Category")
        return fig

app = App(app_ui, server)
'''
with open("app_shiny.py", "w", encoding="utf-8") as f:
    f.write(shiny_app_code)

print("Created app_shiny.py")
print("To run locally, open a terminal in this folder and execute:")
print("shiny run --reload app_shiny.py")

## Slide 9 — Test, interpret, and validate the dashboard
Use this checklist before trusting a dashboard:
- Do filters update every dependent KPI and chart?
- Do totals match a separate Pandas calculation on the same filtered rows?
- Are categories and labels standardized?
- Are missing values or unexpected categories visible?
- Are chart titles, axes, and units clear?

In [ ]:
# A simple independent validation of the cleaned dataset
check_total_sales = df["Sales_INR"].sum()
check_region_total = df.groupby("Region")["Sales_INR"].sum().sum()

print(f"Overall sales: ₹{check_total_sales:,.0f}")
print(f"Sum of region totals: ₹{check_region_total:,.0f}")
print("Totals reconcile:", bool(np.isclose(check_total_sales, check_region_total)))

# Example insight table: top region and category (actual values from this dataset)
top_region = df.groupby("Region")["Sales_INR"].sum().sort_values(ascending=False)
top_category = df.groupby("Category")["Sales_INR"].sum().sort_values(ascending=False)
print("Highest-sales region:", top_region.index[0], f"(₹{top_region.iloc[0]:,.0f})")
print("Highest-sales category:", top_category.index[0], f"(₹{top_category.iloc[0]:,.0f})")

## Slide 10 — Summary and conclusion
### End-to-end workflow
**Business question → data inspection → cleaning → KPI design → charts → interactive controls → validation → decision**

### Key takeaways
- Interactive dashboards help users explore data without repeatedly creating static reports.
- Dash and Shiny for Python both support reactive, user-driven applications.
- Good dashboards require correct data, suitable metrics, readable charts, and tested filters.
- An attractive dashboard is not enough: displayed metrics must be traceable to the source data.

### Exit ticket
In one or two sentences, answer: **What additional filter or visualization would make this sales dashboard more useful, and why?**

## Troubleshooting and delivery notes
- **Excel file not found:** upload `DAV_Dataset_New(7).xlsx` or place it beside this notebook.
- **Dash does not open inline:** try `app.run(debug=False, jupyter_mode="external")`, or run the app from a Python script in a terminal.
- **Shiny app:** keep `app_shiny.py` and the Excel workbook in the same folder; run `shiny run --reload app_shiny.py`.
- **Colab:** browser-based app hosting can require extra configuration. The data cleaning and Plotly cells work directly in the notebook; run Dash/Shiny locally if notebook hosting is unavailable.
- **Data integrity:** duplicate removal is demonstrated for teaching. In production, confirm the business definition of a duplicate before deleting repeated records.